# Toy example: loading and using Qwen2.5-Math-PRM-7B

This notebook demonstrates how to correctly load and use `Qwen/Qwen2.5-Math-PRM-7B` as a process reward model (PRM).

The key idea is:

1. Write the math problem as a chat-style prompt.
2. Split the candidate solution into reasoning steps.
3. Insert `<extra_0>` after each step.
4. Run a single forward pass through the PRM.
5. Read the probability of the positive class at every `<extra_0>` position.

The resulting values are per-step estimates of how good/correct the reasoning trajectory is up to those steps.

Official model card: https://huggingface.co/Qwen/Qwen2.5-Math-PRM-7B


## 0. Installation

Run this once if the needed packages are not already installed.

The PRM is a 7B/8B-class model and usually needs a GPU. Loading it in BF16 is the standard path from the model card.


In [1]:
# !pip install -U "transformers>=4.40.0" accelerate torch


## 1. Imports and model loading

Use `trust_remote_code=True`, because the Qwen PRM uses custom model code.


In [2]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, AutoConfig

# Model paths
base_dir = '/groups/chichengz/tnn/datasets'

qwen_prm_dir    = f"{base_dir}/Qwen2.5-Math-PRM-7B"


qwen_tokenizer = AutoTokenizer.from_pretrained(
    qwen_prm_dir,
    trust_remote_code=True,
)

if qwen_tokenizer.pad_token_id is None:
    qwen_tokenizer.pad_token_id = qwen_tokenizer.eos_token_id
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_config = AutoConfig.from_pretrained(
    qwen_prm_dir, trust_remote_code=True,
)
qwen_config.pad_token_id = qwen_tokenizer.pad_token_id

model = AutoModel.from_pretrained(
    qwen_prm_dir,
    config=qwen_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
).eval()


/home/u20/tnguyen9210/micromamba/envs/vllm1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.84it/s]
Some weights of the model checkpoint at /groups/chichengz/tnn/datasets/Qwen2.5-Math-PRM-7B were not used when initializing Qwen2ForProcessRewardModel: ['lm_head.weight']
- This IS expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification 

## 2. Qwen PRM system prompt

This is the system prompt used in the official model-card example.

Important Python detail: write `\\boxed{}` as `"\\\\boxed{}"` inside a normal Python string, otherwise `\b` may be interpreted as a backspace escape.


In [3]:
QWEN_SYSTEM = (
    "Please reason step by step, and put your final answer "
    "within \\boxed{}."
)

print(QWEN_SYSTEM)


Please reason step by step, and put your final answer within \boxed{}.


## 3. Helper functions

`<extra_0>` is the step separator / reward marker. The PRM assigns a binary classification score at each marker position. We read `P(class = 1)` as the step reward.


In [4]:
def build_qwen_prm_conversation(problem, steps, system=QWEN_SYSTEM):
    """Build the chat-formatted string expected by Qwen2.5-Math-PRM-7B.

    Args:
        problem: Math problem string.
        steps: List of reasoning steps. The last step should contain the final answer.
        system: System prompt.

    Returns:
        A formatted conversation string produced by tokenizer.apply_chat_template.
    """
    if not steps:
        raise ValueError("steps must be a non-empty list.")

    # The official Qwen PRM pattern is:
    #     "<extra_0>".join(steps) + "<extra_0>"
    # This ensures that the final step also receives a PRM score.
    assistant_content = "\n\n".join(
        step.strip() + "<extra_0>" for step in steps
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": problem},
        {"role": "assistant", "content": assistant_content},
    ]

    conversation_str = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return conversation_str


def score_qwen_prm(problem, steps, system=QWEN_SYSTEM, print_conversation=False):
    """Return per-step P(correct) scores from Qwen2.5-Math-PRM-7B.

    The returned list has the same length as `steps`.
    """
    conversation_str = build_qwen_prm_conversation(problem, steps, system=system)

    if print_conversation:
        print("===== Formatted conversation =====")
        print(conversation_str)
        print("==================================")

    input_ids = qwen_tokenizer.encode(
        conversation_str,
        return_tensors="pt",
    ).to(model.device)

    sep_ids = qwen_tokenizer.encode("<extra_0>", add_special_tokens=False)
    if len(sep_ids) != 1:
        raise ValueError(f"Expected <extra_0> to be one token, got token ids: {sep_ids}")
    step_sep_id = sep_ids[0]

    token_mask = input_ids == step_sep_id

    with torch.no_grad():
        # For a single forward scoring pass, we do not need a KV cache.
        outputs = model(input_ids=input_ids, use_cache=False)
        logits = outputs[0]  # shape: (batch_size=1, seq_len, 2)

    probs = F.softmax(logits, dim=-1)
    step_probs = probs[0][token_mask[0]][:, 1]

    if step_probs.numel() != len(steps):
        raise RuntimeError(
            f"Expected {len(steps)} scores, but found {step_probs.numel()} scores. "
            "Check that each step is followed by exactly one <extra_0>."
        )

    return step_probs.detach().cpu().float().tolist()


def print_step_scores(steps, scores):
    for i, (step, score) in enumerate(zip(steps, scores), start=1):
        print(f"Step {i}: P(correct) = {score:.4f}")
        print(step)
        print()


## 4. Toy problem

We compare a correct trajectory and an intentionally wrong trajectory for the same simple equation.


In [5]:
problem = "If 3x + 5 = 17, what is x?"

correct_steps = [
    "We need solve the equation 3x + 5 = 17.",
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 3 gives x = 4.",
    "Therefore, the answer is (\\boxed{4}).",
]

wrong_steps = [
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 2 gives x = 6.",
    "Therefore, the final answer is \\boxed{6}.",
]


## 5. Inspect the exact PRM input

The assistant message should contain `<extra_0>` after every step, including the final answer step.

The full chat template may add tokens such as `<|im_end|>` after the assistant content. That is fine. What matters is that every step ends with `<extra_0>` inside the assistant content.


In [6]:
_ = score_qwen_prm(problem, correct_steps, print_conversation=True)


===== Formatted conversation =====
<|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
If 3x + 5 = 17, what is x?<|im_end|>
<|im_start|>assistant
We need solve the equation 3x + 5 = 17.<extra_0>

Subtracting 5 from both sides gives 3x = 12.<extra_0>

Dividing both sides by 3 gives x = 4.<extra_0>

Therefore, the answer is (\boxed{4}).<extra_0><|im_end|><|endoftext|>


## 6. Score the correct trajectory


In [7]:
correct_scores = score_qwen_prm(problem, correct_steps)
print_step_scores(correct_steps, correct_scores)


Step 1: P(correct) = 1.0000
We need solve the equation 3x + 5 = 17.

Step 2: P(correct) = 1.0000
Subtracting 5 from both sides gives 3x = 12.

Step 3: P(correct) = 1.0000
Dividing both sides by 3 gives x = 4.

Step 4: P(correct) = 1.0000
Therefore, the answer is (\boxed{4}).



## 7. Score the wrong trajectory

The second step is wrong: from `3x = 12`, we should divide by 3, not by 2.

A useful PRM should assign lower scores around or after the first incorrect step.


In [8]:
wrong_scores = score_qwen_prm(problem, wrong_steps)
print_step_scores(wrong_steps, wrong_scores)


Step 1: P(correct) = 0.9922
Subtracting 5 from both sides gives 3x = 12.

Step 2: P(correct) = 0.0152
Dividing both sides by 2 gives x = 6.

Step 3: P(correct) = 0.5898
Therefore, the final answer is \boxed{6}.



## 8. Simple trajectory-level aggregation

Common aggregation choices include:

- `min(scores)`: conservative; a trajectory is only as good as its weakest step.
- `prod(scores)`: treats step correctness as roughly multiplicative.
- `mean(scores)`: smoother, but can hide a single severe error.
- `last(scores)`: uses the final-step score only.

For detecting intermediate mistakes, `min(scores)` is often a useful first diagnostic.


In [9]:
import math

def aggregate_scores(scores):
    return {
        "min": min(scores),
        "mean": sum(scores) / len(scores),
        "prod": math.prod(scores),
        "last": scores[-1],
    }

print("Correct trajectory:")
print(aggregate_scores(correct_scores))

print("\nWrong trajectory:")
print(aggregate_scores(wrong_scores))


Correct trajectory:
{'min': 1.0, 'mean': 1.0, 'prod': 1.0, 'last': 1.0}

Wrong trajectory:
{'min': 0.01519775390625, 'mean': 0.53240966796875, 'prod': 0.008894266560673714, 'last': 0.58984375}


## 9. Common pitfalls

1. **Forgetting the final `<extra_0>`**  
   If the final step does not end with `<extra_0>`, you will not get a reward for the final answer step.

2. **Using the PRM as a generator**  
   This model is for scoring reasoning steps, not generating solutions.

3. **Wrong Python escaping for `\\boxed{}`**  
   Use `"\\\\boxed{4}"` inside a normal Python string.

4. **Bad step segmentation**  
   Each item in `steps` should be a meaningful reasoning step. Do not put the entire solution in one giant step if you want useful process-level feedback.

5. **Interpreting scores as ground truth**  
   PRM scores are learned estimates. They are useful for ranking and diagnostics, but they are not guaranteed proof of correctness.
